# Email Spam Classification: End-to-End NLP Project

# 1. Data Overview

This notebook develops a leakage-safe binary classifier for distinguishing spam from legitimate email. It covers exploratory analysis, hypothesis testing, preprocessing, representation learning, feature selection, model comparison, tuning, threshold selection, independent evaluation, error analysis, and model packaging.

In [ ]:
# Perform the analysis step below.
import pandas as pd
import numpy as np

# Perform the analysis step below.
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer


In [ ]:
# Perform the analysis step below.
import re
from sklearn.model_selection import train_test_split

raw_data = pd.read_csv("data/emails.csv")


def normalize_for_split(text):
    """Creates a grouping key so duplicate emails cannot enter different splits."""
    text = str(text).lower()
    text = re.sub(r"^subject:\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " html ", text)
    text = re.sub(
        r"\b[\w.+-]+\s*@\s*[\w.-]+\s*\.\s*[a-z]{2,}\b",
        " email ",
        text,
    )
    text = re.sub(r"https?\s*:\s*/\s*/|www\s*\.\s*", " url ", text)
    text = re.sub(r"[$€£]\s*\d+(?:[.,]\d+)?", " money ", text)
    text = re.sub(r"\d+", " number ", text)
    text = re.sub(r"[^a-z_\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


raw_data["_split_key"] = (
    raw_data["text"].fillna("").astype(str).map(normalize_for_split)
)
raw_data = raw_data.loc[raw_data["_split_key"].ne("")].copy()

group_label_counts = raw_data.groupby("_split_key")["spam"].nunique()
if (group_label_counts > 1).any():
    raise ValueError("Identical emails have conflicting labels")

group_targets = raw_data.groupby("_split_key")["spam"].first()
development_keys, final_test_keys = train_test_split(
    group_targets.index.to_numpy(),
    test_size=0.15,
    random_state=0,
    stratify=group_targets.to_numpy(),
)

df = (
    raw_data.loc[raw_data["_split_key"].isin(development_keys), ["text", "spam"]]
    .reset_index(drop=True)
)
locked_final_raw = (
    raw_data.loc[raw_data["_split_key"].isin(final_test_keys), ["text", "spam"]]
    .reset_index(drop=True)
)

print("Development pool:", len(df))
print("Locked final test rows before deduplication:", len(locked_final_raw))
df.head()


In [ ]:
# Perform the analysis step below.
df.shape

In [ ]:
# Perform the analysis step below.
df.columns

In [ ]:
# Perform the analysis step below.
df.dtypes

In [ ]:
# Perform the analysis step below.
df.info()

In [ ]:
# Perform the analysis step below.
df.describe(include="all")

In [ ]:
# Perform the analysis step below.
df.isnull().sum()

In [ ]:
# Perform the analysis step below.
df.duplicated().sum()

In [ ]:
# Perform the analysis step below.
df.duplicated(subset="text").sum()

In [ ]:
# Perform the analysis step below.
df["spam"].value_counts()

In [ ]:
# Perform the analysis step below.
(df["spam"].value_counts(normalize=True) * 100).round(2)

In [ ]:
# Perform the analysis step below.
plt.figure(figsize=(6, 4))

sns.countplot(data=df, x="spam")

plt.title("Class distribution")
plt.xlabel("Class: 0 = legitimate, 1 = spam")
plt.ylabel("Number of emails")
plt.show()

In [ ]:
# Perform the analysis step below.
df["text_length"] = df["text"].str.len()

df[["text", "text_length"]].head()

In [ ]:
# Perform the analysis step below.
df["text_length"].describe()

In [ ]:
# Perform the analysis step below.
df.groupby("spam")["text_length"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

In [ ]:
# Perform the analysis step below.
text_length_95 = df["text_length"].quantile(0.95)

plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="text_length",
    bins=50,
    kde=True
)

plt.title("Email-length distribution")
plt.xlabel("Number of characters")
plt.ylabel("Number of emails")
plt.show()

In [ ]:
# Perform the analysis step below.
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df[df["text_length"] <= text_length_95],
    x="text_length",
    hue="spam",
    bins=50,
    kde=True,
    element="step"
)

plt.title("Length of legitimate and spam emails")
plt.xlabel("Number of characters")
plt.ylabel("Number of emails")
plt.show()

In [ ]:
# Perform the analysis step below.
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df[df["text_length"] <= text_length_95],
    x="spam",
    y="text_length"
)

plt.title("Email length by class")
plt.xlabel("Class")
plt.ylabel("Number of characters")
plt.show()

In [ ]:
# Perform the analysis step below.
df["text"].isnull().sum()

In [ ]:
# Perform the analysis step below.
df["text"].fillna("").str.strip().eq("").sum()

In [ ]:
# Perform the analysis step below.
df.loc[df["text_length"].idxmin(), ["text", "spam", "text_length"]]

In [ ]:
# Perform the analysis step below.
df.loc[df["text_length"].idxmax(), ["text", "spam", "text_length"]]

In [ ]:
# Perform the analysis step below.
df["log_text_length"] = np.log1p(df["text_length"])

In [ ]:
# Perform the analysis step below.
sns.histplot(
    data=df,
    x="log_text_length",
    hue="spam",
    kde=True,
    element="step"
)

plt.show()

# 2. Hypothesis Testing

## Hypothesis 1. Legitimate emails are longer than spam

**Hypothesis:** legitimate emails are longer on average than spam messages.

**Test:** compare mean and median length, inspect the `log1p` distribution, and use one-way ANOVA to check whether the difference is unlikely to be random.

In [ ]:
# Perform the analysis step below.
from scipy.stats import f_oneway

normal_length = df.loc[df["spam"] == 0, "log_text_length"]
spam_length = df.loc[df["spam"] == 1, "log_text_length"]

f_statistic, p_value = f_oneway(
    normal_length,
    spam_length
)

print("F-statistic:", f_statistic)
print("P-value:", p_value)

In [ ]:
# Perform the analysis step below.
alpha = 0.05

print(f"F-statistic: {f_statistic:.2f}")
print(f"P-value: {p_value:.4e}")
print(f"Significance level: {alpha}")

if p_value < alpha:
    print("The result is statistically significant.")
else:
    print("The result is not statistically significant.")

**Conclusion:** supported. Legitimate messages average `1,631.92` characters versus `1,317.26` for spam; medians are `1,122` and `693.5`. With `p = 1.196e-33`, the difference is statistically significant. `text_length` is worth testing as a model feature.

## Hypothesis 2. Spam contains distinctive words and word combinations

**Hypothesis:** terms such as `free`, `won`, and `money`, together with promotional bigrams, appear more often in spam.

**Test:** use `CountVectorizer` to find words and bigrams, then compare the share of emails containing each term. Proportions are used because the class sizes differ.

In [ ]:
# Perform the analysis step below.
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

text_for_analysis = df["text"].fillna("").astype(str)
custom_stop_words = list(ENGLISH_STOP_WORDS.union({"subject"}))

vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=custom_stop_words,
    ngram_range=(1, 2),
    min_df=5,
    binary=True
)

X_words = vectorizer.fit_transform(text_for_analysis)
terms = vectorizer.get_feature_names_out()

print("Emails and extracted terms:", X_words.shape)

In [ ]:
# Perform the analysis step below.
spam_mask = df["spam"].eq(1).to_numpy()
normal_mask = df["spam"].eq(0).to_numpy()

spam_frequency = np.asarray(
    X_words[spam_mask].mean(axis=0)
).ravel()

normal_frequency = np.asarray(
    X_words[normal_mask].mean(axis=0)
).ravel()

In [ ]:
# Perform the analysis step below.
words_comparison = pd.DataFrame({
    "term": terms,
    "spam_frequency": spam_frequency,
    "normal_frequency": normal_frequency
})

words_comparison["difference"] = (
    words_comparison["spam_frequency"]
    - words_comparison["normal_frequency"]
)
words_comparison["spam_frequency_pct"] = words_comparison["spam_frequency"] * 100
words_comparison["normal_frequency_pct"] = words_comparison["normal_frequency"] * 100
words_comparison["term_type"] = np.where(
    words_comparison["term"].str.contains(" ", regex=False),
    "Bigram",
    "Word"
)

top_spam_terms = (
    words_comparison
    .sort_values("difference", ascending=False)
    .groupby("term_type", group_keys=False)
    .head(15)
)

top_spam_terms[
    ["term_type", "term", "spam_frequency_pct",
     "normal_frequency_pct", "difference"]
].round(3)

In [ ]:
# Perform the analysis step below.
words_to_check = ["free", "won", "money"]

words_comparison[
    words_comparison["term"].isin(words_to_check)
][
    ["term", "spam_frequency_pct",
     "normal_frequency_pct", "difference"]
].sort_values("difference", ascending=False).round(3)

In [ ]:
# Perform the analysis step below.
top_words = (
    words_comparison[words_comparison["term_type"] == "Word"]
    .nlargest(12, "difference")
    .sort_values("difference")
)
top_bigrams = (
    words_comparison[words_comparison["term_type"] == "Bigram"]
    .nlargest(12, "difference")
    .sort_values("difference")
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.barplot(data=top_words, x="difference", y="term", color="steelblue", ax=axes[0])
axes[0].set_title("Words associated with spam")
axes[0].set_xlabel("Frequency difference: spam minus legitimate")
axes[0].set_ylabel("Word")

sns.barplot(data=top_bigrams, x="difference", y="term", color="darkorange", ax=axes[1])
axes[1].set_title("Bigrams associated with spam")
axes[1].set_xlabel("Frequency difference: spam minus legitimate")
axes[1].set_ylabel("Bigram")

plt.tight_layout()
plt.show()

**Conclusion:** supported. `money` appears in `20.47%` of spam versus `2.00%` of legitimate mail; `free` in `17.47%` versus `10.14%`; and `won` in `2.27%` versus `1.28%`. Other useful signals include `click`, `offer`, `website`, `mailing list`, and `save money`. No single word is sufficient, so the model should combine many textual signals.

## Hypothesis 3. Spam uses more exclamation and question marks

**Hypothesis:** spam uses more `!` and/or `?` to attract attention.

**Test:** count punctuation per message and per 1,000 characters, then compare classes visually and with ANOVA.

In [ ]:
# Perform the analysis step below.
df["exclamation_count"] = text_for_analysis.str.count("!")
df["question_count"] = text_for_analysis.str.count(r"\?")
df["punctuation_count"] = (
    df["exclamation_count"] + df["question_count"]
)

safe_text_length = df["text_length"].clip(lower=1)
df["exclamation_rate"] = df["exclamation_count"] / safe_text_length * 1000
df["question_rate"] = df["question_count"] / safe_text_length * 1000
df["punctuation_rate"] = df["punctuation_count"] / safe_text_length * 1000

In [ ]:
# Perform the analysis step below.
punctuation_summary = df.groupby("spam")[[
    "exclamation_count", "question_count", "punctuation_count",
    "exclamation_rate", "question_rate", "punctuation_rate"
]].agg(["mean", "median"]).round(3)

punctuation_summary

In [ ]:
# Perform the analysis step below.
punctuation_plot = (
    df.groupby("spam")[["exclamation_rate", "question_rate"]]
    .mean()
    .reset_index()
    .melt(id_vars="spam", var_name="sign", value_name="mean_rate")
)
punctuation_plot["class"] = punctuation_plot["spam"].map({
    0: "Legitimate emails",
    1: "Spam"
})
punctuation_plot["sign"] = punctuation_plot["sign"].map({
    "exclamation_rate": "Exclamation marks",
    "question_rate": "Question marks"
})

plt.figure(figsize=(10, 6))
sns.barplot(
    data=punctuation_plot,
    x="sign",
    y="mean_rate",
    hue="class"
)
plt.title("Average punctuation per 1,000 characters")
plt.xlabel("Punctuation type")
plt.ylabel("Count per 1,000 characters")
plt.tight_layout()
plt.show()

In [ ]:
# Perform the analysis step below.
punctuation_tests = []

for column, label in {
    "exclamation_rate": "Exclamation marks",
    "question_rate": "Question marks",
    "punctuation_rate": "All punctuation"
}.items():
    normal_values = np.log1p(df.loc[df["spam"] == 0, column])
    spam_values = np.log1p(df.loc[df["spam"] == 1, column])
    f_statistic, p_value = f_oneway(normal_values, spam_values)

    punctuation_tests.append({
        "feature": label,
        "f_statistic": f_statistic,
        "p_value": p_value,
        "statistically_significant": p_value < 0.05
    })

punctuation_test_results = pd.DataFrame(punctuation_tests)
punctuation_test_results

**Conclusion:** partially supported. Spam contains `2.78` exclamation marks per 1,000 characters versus `0.47` in legitimate mail. Question-mark rates are almost identical: `1.17` versus `1.14`. The combined rate is `3.95` versus `1.61`, with `p < 0.05`. `exclamation_count` and `punctuation_rate` are the more promising features.

## Hypothesis 4. Spam contains more writing noise

**Hypothesis:** spam is written less carefully and contains more observable spelling and punctuation noise.

**Test:** remove URLs, email addresses, and HTML before counting repeated punctuation, excessively repeated letters, missing spaces after punctuation, and very rare tokens. Rates are normalized per 100 words.

**Limitation:** these are proxy signals, not a complete grammar check. A rare word may be a name or technical term; a tool such as LanguageTool would be needed for grammatical validation. Capitalization cannot be evaluated because the dataset is already lowercase.

In [ ]:
# Perform the analysis step below.
import re
from collections import Counter

quality_text = (
    text_for_analysis
    .str.replace(r"(?i)^subject:\s*", "", regex=True)
    .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
    .str.replace(
        r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b",
        " ",
        regex=True
    )
    .str.replace(r"<[^>]+>", " ", regex=True)
)

quality_tokens = quality_text.str.lower().str.findall(r"\b[a-z]{2,}\b")
token_frequency = Counter(
    token for tokens in quality_tokens for token in tokens
)
df["quality_word_count"] = quality_tokens.str.len().clip(lower=1)

In [ ]:
# Perform the analysis step below.
def count_pattern(text, pattern):
    return len(re.findall(pattern, text))


df["repeated_punctuation_count"] = quality_text.apply(
    lambda text: count_pattern(text, r"[!?]{2,}")
)
df["repeated_letter_count"] = quality_text.apply(
    lambda text: count_pattern(
        text,
        r"\b[A-Za-z]*([A-Za-z])\1{2,}[A-Za-z]*\b"
    )
)
df["missing_space_count"] = quality_text.apply(
    lambda text: count_pattern(text, r"[!?][A-Za-z]")
)
df["rare_token_count"] = quality_tokens.apply(
    lambda tokens: sum(token_frequency[token] <= 2 for token in tokens)
)

for feature in [
    "repeated_punctuation",
    "repeated_letter",
    "missing_space",
    "rare_token"
]:
    df[f"{feature}_rate"] = (
        df[f"{feature}_count"] / df["quality_word_count"] * 100
    )

In [ ]:
# Perform the analysis step below.
quality_columns = [
    "repeated_punctuation_rate",
    "repeated_letter_rate",
    "missing_space_rate",
    "rare_token_rate"
]

writing_quality_summary = (
    df.groupby("spam")[quality_columns]
    .agg(["mean", "median"])
    .round(4)
)

writing_quality_summary

In [ ]:
# Perform the analysis step below.
style_columns = [
    "repeated_punctuation_rate",
    "repeated_letter_rate",
    "missing_space_rate"
]

style_plot = (
    df.groupby("spam")[style_columns]
    .mean()
    .reset_index()
    .melt(id_vars="spam", var_name="feature", value_name="mean_rate")
)
style_plot["class"] = style_plot["spam"].map({0: "Legitimate emails", 1: "Spam"})
style_plot["feature"] = style_plot["feature"].map({
    "repeated_punctuation_rate": "Repeated punctuation",
    "repeated_letter_rate": "Repeated letters",
    "missing_space_rate": "Missing space after !/?"
})

rare_plot = (
    df.groupby("spam", as_index=False)["rare_token_rate"]
    .mean()
)
rare_plot["class"] = rare_plot["spam"].map({0: "Legitimate emails", 1: "Spam"})

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=style_plot, x="feature", y="mean_rate", hue="class", ax=axes[0])
axes[0].set_title("Writing-noise indicators")
axes[0].set_xlabel("Feature")
axes[0].set_ylabel("Average count per 100 words")

sns.barplot(data=rare_plot, x="class", y="rare_token_rate", ax=axes[1])
axes[1].set_title("Very rare tokens")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Average count per 100 words")

plt.tight_layout()
plt.show()

In [ ]:
# Perform the analysis step below.
quality_tests = []

for column in quality_columns:
    normal_values = np.log1p(df.loc[df["spam"] == 0, column])
    spam_values = np.log1p(df.loc[df["spam"] == 1, column])
    f_statistic, p_value = f_oneway(normal_values, spam_values)

    quality_tests.append({
        "feature": column,
        "f_statistic": f_statistic,
        "p_value": p_value,
        "statistically_significant": p_value < 0.05
    })

writing_quality_test_results = pd.DataFrame(quality_tests)
writing_quality_test_results

#### Numeric comparison of writing-noise indicators

In [ ]:
# Perform the analysis step below.
quality_labels = {
    "repeated_punctuation_rate": "Repeated !!! and ???",
    "repeated_letter_rate": "Words with repeated letters",
    "missing_space_rate": "Missing space after ! or ?",
    "rare_token_rate": "Very rare tokens"
}

writing_quality_table = (
    df.groupby("spam")[quality_columns]
    .mean()
    .T
    .rename(columns={
        0: "Legitimate emails",
        1: "Spam"
    })
    .rename(index=quality_labels)
)

writing_quality_table["Difference"] = (
    writing_quality_table["Spam"]
    - writing_quality_table["Legitimate emails"]
)
writing_quality_table.index.name = "Indicator per 100 words"

writing_quality_table.round(4)

**Conclusion:** partially supported. Per 100 words, spam contains `0.3780` repeated-letter tokens versus `0.1152`, `0.0904` missing-space cases versus `0.0000`, and `6.2799` rare tokens versus `1.4196`. Each difference has `p < 0.05`. This supports the presence of more textual noise, but does not prove that spam has more grammatical errors.

## Hypothesis 5. Spam contains more links

**Hypothesis:** spam contains links more often and has more links relative to text length.

**Test:** detect standard and space-obfuscated `http://` and `www.` patterns, calculate prevalence and links per 100 words, and compare classes using ANOVA.

In [ ]:
# Perform the analysis step below.
text_lower = text_for_analysis.str.lower()
hypothesis_word_count = (
    text_lower.str.findall(r"\b[a-z]+\b").str.len().clip(lower=1)
)
url_pattern = r"https?\s*:\s*/\s*/|www\s*\.\s*[a-z0-9]"

df["url_count"] = text_lower.str.count(url_pattern)
df["url_rate"] = df["url_count"] / hypothesis_word_count * 100
df["has_url"] = df["url_count"].gt(0).astype(int)

In [ ]:
# Perform the analysis step below.
url_summary = df.groupby("spam").agg(
    emails=("spam", "size"),
    mean_url_count=("url_count", "mean"),
    mean_url_rate=("url_rate", "mean"),
    emails_with_url=("has_url", "sum"),
    share_with_url=("has_url", "mean")
).reset_index()
url_summary["share_with_url_pct"] = url_summary["share_with_url"] * 100
url_summary.round(4)

In [ ]:
# Perform the analysis step below.
plt.figure(figsize=(7, 5))
sns.barplot(data=url_summary, x="spam", y="share_with_url_pct", color="steelblue")
plt.title("Share of emails containing links")
plt.xlabel("Class: 0 = legitimate, 1 = spam")
plt.ylabel("Emails containing a link, %")
plt.show()

In [ ]:
# Perform the analysis step below.
normal_url_rate = np.log1p(df.loc[df["spam"] == 0, "url_rate"])
spam_url_rate = np.log1p(df.loc[df["spam"] == 1, "url_rate"])
url_f_statistic, url_p_value = f_oneway(normal_url_rate, spam_url_rate)

print(f"F-statistic: {url_f_statistic:.2f}")
print(f"P-value: {url_p_value:.4e}")

**Conclusion:** supported. Links occur in `29.02%` of spam versus `16.35%` of legitimate mail. Spam has `0.5610` links per 100 words versus `0.1909`, with `p = 1.2318e-44`. `url_count`, `url_rate`, and `has_url` are reasonable candidate features.

## Hypothesis 6. Spam refers to money and financial benefit more often

**Hypothesis:** spam contains more currency symbols and vocabulary related to money, prices, credit, prizes, and savings.

**Test:** count `$`, `€`, `£`, and a predefined list of financial expressions; normalize per 100 words and compare the two classes with ANOVA.

In [ ]:
# Perform the analysis step below.
money_term_pattern = (
    r"\b(?:money|cash|dollars?|usd|price|prices|discount|discounts|"
    r"offer|offers|credit|loan|loans|income|profit|profits|save|saving|"
    r"win|winner|won|free)\b"
)
df["currency_count"] = text_lower.str.count(r"[$€£]")
df["money_term_count"] = text_lower.str.count(money_term_pattern)
df["money_expression_count"] = (
    df["currency_count"] + df["money_term_count"]
)
df["money_expression_rate"] = (
    df["money_expression_count"] / hypothesis_word_count * 100
)
df["has_money_expression"] = df["money_expression_count"].gt(0).astype(int)

In [ ]:
# Perform the analysis step below.
money_summary = df.groupby("spam").agg(
    emails=("spam", "size"),
    mean_money_count=("money_expression_count", "mean"),
    mean_money_rate=("money_expression_rate", "mean"),
    emails_with_money=("has_money_expression", "sum"),
    share_with_money=("has_money_expression", "mean")
).reset_index()
money_summary["share_with_money_pct"] = money_summary["share_with_money"] * 100
money_summary.round(4)

In [ ]:
# Perform the analysis step below.
plt.figure(figsize=(7, 5))
sns.barplot(data=money_summary, x="spam", y="share_with_money_pct", color="darkorange")
plt.title("Share mentioning money or financial benefit")
plt.xlabel("Class: 0 = legitimate, 1 = spam")
plt.ylabel("Share of emails, %")
plt.show()

In [ ]:
# Perform the analysis step below.
normal_money_rate = np.log1p(df.loc[df["spam"] == 0, "money_expression_rate"])
spam_money_rate = np.log1p(df.loc[df["spam"] == 1, "money_expression_rate"])
money_f_statistic, money_p_value = f_oneway(normal_money_rate, spam_money_rate)

print(f"F-statistic: {money_f_statistic:.2f}")
print(f"P-value: {money_p_value:.4e}")

**Conclusion:** supported. Financial expressions appear in `70.91%` of spam versus `34.82%` of legitimate messages. The rate is `2.5287` per 100 words in spam and `0.5132` in legitimate mail. With `p = 6.2055e-287`, the difference is statistically significant.

## Hypothesis 7. Spam contains more digits

**Hypothesis:** spam contains more digits because it frequently mentions prices, discounts, prizes, and phone numbers.

**Test:** count digits, normalize per 100 characters, compare the proportion of messages containing at least one digit, and run ANOVA.

In [ ]:
# Perform the analysis step below.
safe_text_length = text_for_analysis.str.len().clip(lower=1)
df["digit_count"] = text_for_analysis.str.count(r"\d")
df["digit_rate"] = df["digit_count"] / safe_text_length * 100
df["has_digit"] = df["digit_count"].gt(0).astype(int)

In [ ]:
# Perform the analysis step below.
digit_summary = df.groupby("spam").agg(
    emails=("spam", "size"),
    mean_digit_count=("digit_count", "mean"),
    mean_digit_rate=("digit_rate", "mean"),
    median_digit_rate=("digit_rate", "median"),
    emails_with_digit=("has_digit", "sum"),
    share_with_digit=("has_digit", "mean")
).reset_index()
digit_summary["share_with_digit_pct"] = digit_summary["share_with_digit"] * 100
digit_summary.round(4)

In [ ]:
# Perform the analysis step below.
plt.figure(figsize=(7, 5))
sns.barplot(data=digit_summary, x="spam", y="mean_digit_rate", color="seagreen")
plt.title("Average digits per 100 characters")
plt.xlabel("Class: 0 = legitimate, 1 = spam")
plt.ylabel("Digits per 100 characters")
plt.show()

In [ ]:
# Perform the analysis step below.
normal_digit_rate = np.log1p(df.loc[df["spam"] == 0, "digit_rate"])
spam_digit_rate = np.log1p(df.loc[df["spam"] == 1, "digit_rate"])
digit_f_statistic, digit_p_value = f_oneway(normal_digit_rate, spam_digit_rate)

print(f"F-statistic: {digit_f_statistic:.2f}")
print(f"P-value: {digit_p_value:.4e}")

**Conclusion:** not supported; the direction is opposite to the hypothesis. Legitimate email contains `2.6347` digits per 100 characters versus `1.9221` in spam. At least one digit occurs in `87.61%` of legitimate messages and `76.61%` of spam. The difference is significant (`p = 1.1495e-56`).

## Hypothesis 8. Spam contains more calls to action

**Hypothesis:** spam more often includes phrases such as `click here`, `buy now`, `order now`, and `subscribe`.

**Test:** count a predefined list of call-to-action expressions per 100 words, compare prevalence, and run ANOVA.

In [ ]:
# Perform the analysis step below.
cta_pattern = (
    r"\b(?:click\s+here|buy\s+now|order\s+now|call\s+now|"
    r"limited\s+offer|act\s+now|apply\s+now|contact\s+us|"
    r"sign\s+up|learn\s+more|visit|subscribe)\b"
)
df["cta_count"] = text_lower.str.count(cta_pattern)
df["cta_rate"] = df["cta_count"] / hypothesis_word_count * 100
df["has_cta"] = df["cta_count"].gt(0).astype(int)

In [ ]:
# Perform the analysis step below.
cta_summary = df.groupby("spam").agg(
    emails=("spam", "size"),
    mean_cta_count=("cta_count", "mean"),
    mean_cta_rate=("cta_rate", "mean"),
    emails_with_cta=("has_cta", "sum"),
    share_with_cta=("has_cta", "mean")
).reset_index()
cta_summary["share_with_cta_pct"] = cta_summary["share_with_cta"] * 100
cta_summary.round(4)

In [ ]:
# Perform the analysis step below.
plt.figure(figsize=(7, 5))
sns.barplot(data=cta_summary, x="spam", y="share_with_cta_pct", color="mediumpurple")
plt.title("Share containing calls to action")
plt.xlabel("Class: 0 = legitimate, 1 = spam")
plt.ylabel("Share of emails, %")
plt.show()

In [ ]:
# Perform the analysis step below.
normal_cta_rate = np.log1p(df.loc[df["spam"] == 0, "cta_rate"])
spam_cta_rate = np.log1p(df.loc[df["spam"] == 1, "cta_rate"])
cta_f_statistic, cta_p_value = f_oneway(normal_cta_rate, spam_cta_rate)

print(f"F-statistic: {cta_f_statistic:.2f}")
print(f"P-value: {cta_p_value:.4e}")

**Conclusion:** supported. Calls to action occur in `28.95%` of spam versus `10.60%` of legitimate mail. The rate is `0.4283` versus `0.0898` per 100 words, with `p = 4.3341e-92`. `cta_count`, `cta_rate`, and `has_cta` are reasonable candidate features.

# 3. Data Cleaning and Preprocessing

After hypothesis testing, we audit data quality, apply conservative text normalization, and compare the same baseline before and after cleaning.

In [ ]:
# Perform the analysis step below.
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)


## 3.1 Data-quality checks

In [ ]:
# Perform the analysis step below.
data = df[["text", "spam"]].copy()
clean_data = data.copy()

clean_data.head()


In [ ]:
# Perform the analysis step below.
data_quality_summary = pd.Series({
    "rows": len(clean_data),
    "columns": clean_data.shape[1],
    "missing_text": clean_data["text"].isna().sum(),
    "missing_target": clean_data["spam"].isna().sum(),
    "empty_text": clean_data["text"].fillna("").str.strip().eq("").sum(),
    "full_duplicates": clean_data.duplicated().sum(),
    "text_duplicates": clean_data.duplicated(subset="text").sum()
})

data_quality_summary

In [ ]:
# Perform the analysis step below.
text_label_counts = clean_data.groupby("text")["spam"].nunique()
conflicting_texts = text_label_counts[text_label_counts > 1]

print("Texts with conflicting labels:", len(conflicting_texts))

## 3.2 Mandatory cleaning

In [ ]:
# Perform the analysis step below.
clean_data["original_text"] = clean_data["text"].fillna("").astype(str)
clean_data = clean_data.loc[
    clean_data["original_text"].str.strip().ne("")
].copy()

rows_before_duplicates = len(clean_data)
clean_data = (
    clean_data
    .drop_duplicates(subset="original_text")
    .reset_index(drop=True)
)

print("Rows before deduplication:", rows_before_duplicates)
print("Exact duplicates removed:", rows_before_duplicates - len(clean_data))
print("Rows remaining:", len(clean_data))

## 3.3 Clean-text representation

The raw message is preserved in `original_text` and the normalized version in `clean_text`. Technical patterns are replaced with semantic tokens, text is lowercased, and whitespace is normalized.

In [ ]:
# Perform the analysis step below.
def clean_email_text(text):
    text = str(text).lower()

    # Perform the analysis step below.
    text = re.sub(r"^subject:\s*", "", text, flags=re.IGNORECASE)

    # Perform the analysis step below.
    text = re.sub(r"<[^>]+>", " html " , text)
    text = re.sub(
        r"\b[\w.+-]+\s*@\s*[\w.-]+\s*\.\s*[a-z]{2,}\b",
        " email ",
        text
    )
    text = re.sub(r"https?\s*:\s*/\s*/", " url ", text)
    text = re.sub(r"www\s*\.\s*", " url ", text)
    text = re.sub(r"[$€£]\s*\d+(?:[.,]\d+)?", " money ", text)
    text = re.sub(r"\d+", " number ", text)

    # Perform the analysis step below.
    text = re.sub(r"[^a-z_\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# Perform the analysis step below.
clean_data["clean_text"] = clean_data["original_text"].apply(clean_email_text)

clean_data[["original_text", "clean_text", "spam"]].head(3)

In [ ]:
# Perform the analysis step below.
cleaning_checks = pd.Series({
    "rows": len(clean_data),
    "empty_clean_text": clean_data["clean_text"].str.strip().eq("").sum(),
    "clean_text_duplicates": clean_data.duplicated(subset="clean_text").sum(),
    "missing_clean_text": clean_data["clean_text"].isna().sum()
})

cleaning_checks

In [ ]:
# Perform the analysis step below.
clean_label_counts = clean_data.groupby("clean_text")["spam"].nunique()
clean_conflicts = clean_label_counts[clean_label_counts > 1]

print("Normalized texts with conflicting labels:", len(clean_conflicts))

In [ ]:
# Perform the analysis step below.
rows_before_clean_duplicates = len(clean_data)
clean_data = (
    clean_data
    .drop_duplicates(subset="clean_text")
    .reset_index(drop=True)
)

print(
    "Duplicates removed after cleaning:",
    rows_before_clean_duplicates - len(clean_data)
)
print("Final number of emails:", len(clean_data))

## 3.4 Decision on extreme values

Very long emails and high counts of digits or URLs can be genuine observations. Rows are not removed automatically; plausible transformations are compared separately with cross-validation.

In [ ]:
# Perform the analysis step below.
class_distribution = pd.DataFrame({
    "count": clean_data["spam"].value_counts(),
    "percent": (clean_data["spam"].value_counts(normalize=True) * 100).round(2)
}).sort_index()

class_distribution

## 3.5 Baseline before and after cleaning

Both variants use the same train/validation rows, TF-IDF, and Logistic Regression. Only the input text changes. TF-IDF is fitted inside the pipeline on training data only; the locked final test remains untouched.

In [ ]:
# Perform the analysis step below.
train_index, validation_index = train_test_split(
    np.arange(len(clean_data)),
    test_size=0.20,
    random_state=0,
    stratify=clean_data["spam"]
)

y_train = clean_data.loc[train_index, "spam"]
y_validation = clean_data.loc[validation_index, "spam"]

print("Train:", len(train_index))
print("Validation:", len(validation_index))


In [ ]:
# Perform the analysis step below.
def build_baseline_pipeline():
    return Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.98,
                max_features=30000,
                sublinear_tf=True
            )
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=0
            )
        )
    ])

In [ ]:
# Perform the analysis step below.
def evaluate_text_version(stage, text_column):
    model = build_baseline_pipeline()
    X_train = clean_data.loc[train_index, text_column]
    X_validation = clean_data.loc[validation_index, text_column]

    model.fit(X_train, y_train)
    result = {"stage": stage}

    for sample, X, y in [
        ("train", X_train, y_train),
        ("validation", X_validation, y_validation)
    ]:
        predictions = model.predict(X)
        probabilities = model.predict_proba(X)[:, 1]
        result[f"{sample}_accuracy"] = accuracy_score(y, predictions)
        result[f"{sample}_precision"] = precision_score(
            y, predictions, zero_division=0
        )
        result[f"{sample}_recall"] = recall_score(
            y, predictions, zero_division=0
        )
        result[f"{sample}_f1"] = f1_score(y, predictions, zero_division=0)
        result[f"{sample}_roc_auc"] = roc_auc_score(y, probabilities)
        result[f"{sample}_pr_auc"] = average_precision_score(y, probabilities)

    result["roc_auc_gap"] = (
        result["train_roc_auc"] - result["validation_roc_auc"]
    )
    return model, result


In [ ]:
# Perform the analysis step below.
raw_model, raw_result = evaluate_text_version(
    "Before text cleaning",
    "original_text"
)
clean_model, clean_result = evaluate_text_version(
    "After text cleaning",
    "clean_text"
)

cleaning_results = pd.DataFrame([raw_result, clean_result])
cleaning_results.round(4)

In [ ]:
# Perform the analysis step below.
validation_metrics = [
    "validation_accuracy",
    "validation_precision",
    "validation_recall",
    "validation_f1",
    "validation_roc_auc",
    "validation_pr_auc",
    "roc_auc_gap"
]

cleaning_changes = pd.DataFrame({
    "metric": validation_metrics,
    "before_cleaning": cleaning_results.loc[0, validation_metrics].values,
    "after_cleaning": cleaning_results.loc[1, validation_metrics].values
})
cleaning_changes["change"] = (
    cleaning_changes["after_cleaning"]
    - cleaning_changes["before_cleaning"]
)

cleaning_changes.round(4)


In [ ]:
# Perform the analysis step below.
plot_metrics = [
    "validation_accuracy", "validation_precision", "validation_recall",
    "validation_f1", "validation_roc_auc", "validation_pr_auc"
]
plot_data = cleaning_results[["stage"] + plot_metrics].melt(
    id_vars="stage",
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(13, 6))
sns.barplot(data=plot_data, x="metric", y="score", hue="stage")
plt.title("Baseline before and after text cleaning")
plt.xlabel("Metric")
plt.ylabel("Value")
plt.ylim(0.90, 1.01)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 3.6 Cleaning conclusion

Validation F1 changed from **0.9745** to **0.9703**, while PR-AUC changed from **0.9988** to **0.9983**. Cleaning did not improve the baseline score, but standardized the input, removed duplicate forms, and created a reproducible foundation. The locked final test was not used.

In [ ]:
# Perform the analysis step below.
clean_data = clean_data[["original_text", "clean_text", "spam"]].copy()

print("Prepared data are stored in clean_data")
print("Data shape:", clean_data.shape)
clean_data.head()

# 4. Extreme-Value Analysis and Treatment Experiments

In [ ]:
# Perform the analysis step below.
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    FunctionTransformer,
    KBinsDiscretizer,
    PowerTransformer,
    RobustScaler,
    StandardScaler
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    make_scorer
)
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

## 4.1 Numerical indicators

Create candidate numerical features motivated by the hypotheses. Raw messages remain in `original_text`, while TF-IDF uses `clean_text`.

In [ ]:
# Perform the analysis step below.
model_data = clean_data.copy()
feature_text = model_data["original_text"].fillna("").astype(str)
feature_text_lower = feature_text.str.lower()

model_data["text_length"] = feature_text.str.len()
model_data["word_count"] = feature_text_lower.str.findall(r"\b[a-z]+\b").str.len()
model_data["exclamation_count"] = feature_text.str.count("!")
model_data["question_count"] = feature_text.str.count(r"\?")
model_data["url_count"] = feature_text_lower.str.count(
    r"https?\s*:\s*/\s*/|www\s*\.\s*[a-z0-9]"
)
model_data["digit_count"] = feature_text.str.count(r"\d")

money_pattern = (
    r"\b(?:money|cash|dollars?|usd|price|prices|discount|discounts|"
    r"offer|offers|credit|loan|loans|income|profit|profits|save|saving|"
    r"win|winner|won|free)\b"
)
model_data["money_expression_count"] = (
    feature_text.str.count(r"[$€£]")
    + feature_text_lower.str.count(money_pattern)
)

cta_pattern = (
    r"\b(?:click\s+here|buy\s+now|order\s+now|call\s+now|"
    r"limited\s+offer|act\s+now|apply\s+now|contact\s+us|"
    r"sign\s+up|learn\s+more|visit|subscribe)\b"
)
model_data["cta_count"] = feature_text_lower.str.count(cta_pattern)

for feature in ["url", "digit", "money_expression", "cta"]:
    model_data[f"has_{feature}"] = (
        model_data[f"{feature}_count"] > 0
    ).astype(int)

numeric_features = [
    "text_length",
    "word_count",
    "exclamation_count",
    "question_count",
    "url_count",
    "digit_count",
    "money_expression_count",
    "cta_count"
]

binary_features = [
    "has_url",
    "has_digit",
    "has_money_expression",
    "has_cta"
]

model_data[["clean_text", "spam"] + numeric_features + binary_features].head()

## 4.2 Inspecting extreme values

IQR identifies potential outliers and the 99th percentile highlights truly extreme observations. For zero-inflated features, IQR may flag every non-zero value, so examples and domain meaning must guide the decision.

In [ ]:
# Perform the analysis step below.
outlier_rows = []

for feature in numeric_features:
    values = model_data[feature]
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    iqr_upper = q3 + 1.5 * iqr
    p99 = values.quantile(0.99)

    outlier_rows.append({
        "feature": feature,
        "median": values.median(),
        "mean": values.mean(),
        "iqr_upper": iqr_upper,
        "iqr_outliers": (values > iqr_upper).sum(),
        "iqr_outliers_pct": (values > iqr_upper).mean() * 100,
        "p99": p99,
        "above_p99": (values > p99).sum(),
        "max": values.max(),
        "skew": values.skew(),
        "log1p_skew": np.log1p(values).skew()
    })

outlier_summary = pd.DataFrame(outlier_rows)
outlier_summary.round(3)

In [ ]:
# Perform the analysis step below.
longest_emails = model_data.nlargest(10, "text_length")[
    ["spam", "text_length", "word_count", "url_count", "digit_count", "original_text"]
].copy()

longest_emails["preview"] = (
    longest_emails["original_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 180)
)

longest_emails.drop(columns="original_text")

In [ ]:
# Perform the analysis step below.
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.histplot(model_data["text_length"], bins=60, ax=axes[0, 0])
axes[0, 0].set_title("Email length before transformation")

sns.histplot(np.log1p(model_data["text_length"]), bins=60, ax=axes[0, 1])
axes[0, 1].set_title("Email length after log1p")

sns.histplot(model_data["digit_count"], bins=60, ax=axes[1, 0])
axes[1, 0].set_title("Digit count before transformation")

sns.histplot(np.log1p(model_data["digit_count"]), bins=60, ax=axes[1, 1])
axes[1, 1].set_title("Digit count after log1p")

plt.tight_layout()
plt.show()

**Preliminary decision:** do not remove extreme emails because they include real corporate newsletters, tables, and genuine spam. Compare transformations, clipping, and robust scaling instead of trimming rows.

## 4.3 Leakage-safe comparison

All variants use the same training data, Logistic Regression, TF-IDF, and folds. Selection is based on mean CV F1, with ROC-AUC, recall, variability, and train-validation gap as supporting evidence.

In [ ]:
# Perform the analysis step below.
class PercentileCapper(BaseEstimator, TransformerMixin):
    def __init__(self, quantile=0.99):
        self.quantile = quantile

    def fit(self, X, y=None):
        X_array = np.asarray(X, dtype=float)
        self.upper_bounds_ = np.nanquantile(
            X_array,
            self.quantile,
            axis=0
        )
        return self

    def transform(self, X):
        X_array = np.asarray(X, dtype=float)
        return np.minimum(X_array, self.upper_bounds_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

In [ ]:
# Perform the analysis step below.
def make_tfidf():
    return TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=30000,
        sublinear_tf=True
    )


numeric_transformers = {
    "Raw + StandardScaler": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]),
    "Log1p + StandardScaler": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", StandardScaler())
    ]),
    "Yeo-Johnson": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("power", PowerTransformer(method="yeo-johnson", standardize=True))
    ]),
    "Raw + RobustScaler": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ]),
    "P99 cap + Log1p + StandardScaler": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("capper", PercentileCapper(quantile=0.99)),
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", StandardScaler())
    ]),
    "Quantile bins": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("bins", KBinsDiscretizer(
            n_bins=5,
            encode="onehot",
            strategy="quantile",
            subsample=None
        ))
    ])
}


def make_model_pipeline(numeric_transformer=None):
    transformers = [("text", make_tfidf(), "clean_text")]

    if numeric_transformer is not None:
        transformers.extend([
            ("numeric", numeric_transformer, numeric_features),
            ("binary", "passthrough", binary_features)
        ])

    preprocessing = ColumnTransformer(transformers=transformers)

    return Pipeline([
        ("preprocessing", preprocessing),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=0
        ))
    ])

In [ ]:
# Perform the analysis step below.
feature_columns = ["clean_text"] + numeric_features + binary_features

X_train_out = model_data.loc[train_index, feature_columns].reset_index(drop=True)
X_validation_out = model_data.loc[validation_index, feature_columns].reset_index(drop=True)
y_train_out = model_data.loc[train_index, "spam"].reset_index(drop=True)
y_validation_out = model_data.loc[validation_index, "spam"].reset_index(drop=True)

X_text_train = X_train_out["clean_text"]
X_text_validation = X_validation_out["clean_text"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

experiment_pipelines = {"TF-IDF baseline": make_model_pipeline(None)}
experiment_pipelines.update({
    name: make_model_pipeline(transformer)
    for name, transformer in numeric_transformers.items()
})

print("Train:", X_train_out.shape)
print("Validation:", X_validation_out.shape)
print("Experiments:", len(experiment_pipelines))


In [ ]:
# Perform the analysis step below.
experiment_rows = []

for name, pipeline in experiment_pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train_out,
        y_train_out,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=2
    )

    experiment_rows.append({
        "experiment": name,
        "cv_accuracy": scores["test_accuracy"].mean(),
        "cv_precision": scores["test_precision"].mean(),
        "cv_recall": scores["test_recall"].mean(),
        "cv_f1": scores["test_f1"].mean(),
        "cv_f1_std": scores["test_f1"].std(),
        "cv_roc_auc": scores["test_roc_auc"].mean(),
        "cv_roc_auc_std": scores["test_roc_auc"].std(),
        "cv_pr_auc": scores["test_pr_auc"].mean(),
        "cv_pr_auc_std": scores["test_pr_auc"].std(),
        "train_roc_auc": scores["train_roc_auc"].mean(),
        "roc_auc_gap": (
            scores["train_roc_auc"].mean()
            - scores["test_roc_auc"].mean()
        )
    })

outlier_experiment_results = (
    pd.DataFrame(experiment_rows)
    .sort_values(["cv_f1", "cv_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

outlier_experiment_results.round(4)


In [ ]:
# Perform the analysis step below.
plot_results = outlier_experiment_results.melt(
    id_vars="experiment",
    value_vars=["cv_f1", "cv_roc_auc"],
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(12, 7))
sns.barplot(
    data=plot_results,
    y="experiment",
    x="score",
    hue="metric"
)
plt.title("Cross-validation of extreme-value treatments")
plt.xlabel("Mean score across 5 folds")
plt.ylabel("Variant")
plt.xlim(0.90, 1.00)
plt.tight_layout()
plt.show()

## 4.4 Validation check

Validation did not participate in selecting the treatment. It is used once to compare the TF-IDF baseline with the best numerical-feature variant chosen through cross-validation.

In [ ]:
# Perform the analysis step below.
overall_best_name = outlier_experiment_results.loc[0, "experiment"]
best_numeric_name = (
    outlier_experiment_results
    .query("experiment != 'TF-IDF baseline'")
    .iloc[0]["experiment"]
)

final_candidates = {
    "TF-IDF baseline": experiment_pipelines["TF-IDF baseline"],
    f"Best numeric: {best_numeric_name}": experiment_pipelines[best_numeric_name]
}

final_rows = []
for name, pipeline in final_candidates.items():
    pipeline.fit(X_train_out, y_train_out)
    train_prediction = pipeline.predict(X_train_out)
    validation_prediction = pipeline.predict(X_validation_out)
    train_score = pipeline.predict_proba(X_train_out)[:, 1]
    validation_score = pipeline.predict_proba(X_validation_out)[:, 1]
    final_rows.append({
        "model": name,
        "train_accuracy": accuracy_score(y_train_out, train_prediction),
        "validation_accuracy": accuracy_score(
            y_validation_out, validation_prediction
        ),
        "validation_precision": precision_score(
            y_validation_out, validation_prediction, zero_division=0
        ),
        "validation_recall": recall_score(
            y_validation_out, validation_prediction, zero_division=0
        ),
        "validation_f1": f1_score(
            y_validation_out, validation_prediction, zero_division=0
        ),
        "train_roc_auc": roc_auc_score(y_train_out, train_score),
        "validation_roc_auc": roc_auc_score(
            y_validation_out, validation_score
        ),
        "validation_pr_auc": average_precision_score(
            y_validation_out, validation_score
        ),
        "roc_auc_gap": (
            roc_auc_score(y_train_out, train_score)
            - roc_auc_score(y_validation_out, validation_score)
        )
    })

final_outlier_comparison = pd.DataFrame(final_rows)
print("Best overall variant:", overall_best_name)
print("Best numerical-feature variant:", best_numeric_name)
final_outlier_comparison.round(4)


## 4.5 Conclusion

- Plain `TF-IDF baseline` remained best: CV F1 **0.9738**, ROC-AUC **0.9993**, PR-AUC **0.9979**.
- The best numerical variant, `Raw + StandardScaler`, reached CV F1 **0.9357**.
- On validation, TF-IDF achieved F1 **0.9703** versus **0.9451** for the numerical variant.
- `log1p`, Yeo–Johnson, RobustScaler, clipping, and quantile bins did not improve the baseline.

Extreme values appear to be genuine messages, so no rows are removed and only TF-IDF proceeds to the next stage.

# 5. Feature Engineering: Text Representation

Compare multiple ways to convert messages into numerical features. Every experiment uses the same Logistic Regression, folds, and training rows, so differences are attributable to the representation.

## 5.1 Candidate representations

- CountVectorizer with unigrams;
- CountVectorizer with unigrams and bigrams;
- binary bag of words;
- word TF-IDF with unigrams;
- word TF-IDF with unigrams and bigrams;
- character TF-IDF;
- combined word and character TF-IDF.

Select by mean CV F1 and also review recall, ROC-AUC, PR-AUC, fold variability, and train-validation gap.

In [ ]:
# Perform the analysis step below.
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import FeatureUnion


def make_logistic_regression():
    return LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=0
    )

In [ ]:
# Perform the analysis step below.
text_representations = {
    "Count word (1, 1)": CountVectorizer(
        ngram_range=(1, 1),
        min_df=2,
        max_df=0.98,
        max_features=30000
    ),
    "Count word (1, 2)": CountVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=30000
    ),
    "Binary word (1, 2)": CountVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=30000,
        binary=True
    ),
    "TF-IDF word (1, 1)": TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=2,
        max_df=0.98,
        max_features=30000,
        sublinear_tf=True
    ),
    "TF-IDF word (1, 2)": TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=30000,
        sublinear_tf=True
    ),
    "TF-IDF char (3, 5)": TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=30000,
        sublinear_tf=True
    ),
    "TF-IDF word + char": FeatureUnion([
        ("word", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            max_features=20000,
            sublinear_tf=True
        )),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=30000,
            sublinear_tf=True
        ))
    ])
}

text_feature_pipelines = {
    name: Pipeline([
        ("features", representation),
        ("model", make_logistic_regression())
    ])
    for name, representation in text_representations.items()
}

print("Number of variants:", len(text_feature_pipelines))

In [ ]:
# Perform the analysis step below.
X_text_train = X_train_out["clean_text"]
X_text_validation = X_validation_out["clean_text"]

print("Train:", X_text_train.shape)
print("Validation:", X_text_validation.shape)

In [ ]:
# Perform the analysis step below.
text_feature_rows = []

for name, pipeline in text_feature_pipelines.items():
    scores = cross_validate(
        pipeline,
        X_text_train,
        y_train_out,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=2
    )

    text_feature_rows.append({
        "representation": name,
        "cv_accuracy": scores["test_accuracy"].mean(),
        "cv_precision": scores["test_precision"].mean(),
        "cv_recall": scores["test_recall"].mean(),
        "cv_f1": scores["test_f1"].mean(),
        "cv_f1_std": scores["test_f1"].std(),
        "cv_roc_auc": scores["test_roc_auc"].mean(),
        "cv_roc_auc_std": scores["test_roc_auc"].std(),
        "cv_pr_auc": scores["test_pr_auc"].mean(),
        "cv_pr_auc_std": scores["test_pr_auc"].std(),
        "train_roc_auc": scores["train_roc_auc"].mean(),
        "roc_auc_gap": (
            scores["train_roc_auc"].mean()
            - scores["test_roc_auc"].mean()
        )
    })

text_feature_results = (
    pd.DataFrame(text_feature_rows)
    .sort_values(["cv_f1", "cv_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

text_feature_results.round(4)


In [ ]:
# Perform the analysis step below.
text_feature_plot = text_feature_results.melt(
    id_vars="representation",
    value_vars=["cv_f1", "cv_roc_auc"],
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(12, 7))
sns.barplot(
    data=text_feature_plot,
    y="representation",
    x="score",
    hue="metric"
)
plt.title("Cross-validation of text representations")
plt.xlabel("Mean score across 5 folds")
plt.ylabel("Representation")
plt.xlim(0.90, 1.00)
plt.tight_layout()
plt.show()

## 5.2 Validation check

The representation was selected without validation. Compare the current word TF-IDF `(1, 2)` baseline with the cross-validation winner.

In [ ]:
# Perform the analysis step below.
best_text_representation = text_feature_results.loc[0, "representation"]
validation_candidates = {
    "Current TF-IDF word (1, 2)": text_feature_pipelines["TF-IDF word (1, 2)"],
    f"Winner: {best_text_representation}": text_feature_pipelines[best_text_representation]
}

text_validation_rows = []
for name, pipeline in validation_candidates.items():
    pipeline.fit(X_text_train, y_train_out)
    train_prediction = pipeline.predict(X_text_train)
    validation_prediction = pipeline.predict(X_text_validation)
    train_score = pipeline.predict_proba(X_text_train)[:, 1]
    validation_score = pipeline.predict_proba(X_text_validation)[:, 1]
    text_validation_rows.append({
        "model": name,
        "train_accuracy": accuracy_score(y_train_out, train_prediction),
        "validation_accuracy": accuracy_score(
            y_validation_out, validation_prediction
        ),
        "validation_precision": precision_score(
            y_validation_out, validation_prediction
        ),
        "validation_recall": recall_score(
            y_validation_out, validation_prediction
        ),
        "validation_f1": f1_score(
            y_validation_out, validation_prediction
        ),
        "train_roc_auc": roc_auc_score(y_train_out, train_score),
        "validation_roc_auc": roc_auc_score(
            y_validation_out, validation_score
        ),
        "validation_pr_auc": average_precision_score(
            y_validation_out, validation_score
        ),
        "roc_auc_gap": (
            roc_auc_score(y_train_out, train_score)
            - roc_auc_score(y_validation_out, validation_score)
        )
    })

text_feature_validation = pd.DataFrame(text_validation_rows)
best_text_pipeline = text_feature_pipelines[best_text_representation]
print("Best representation by CV F1:", best_text_representation)
text_feature_validation.round(4)


## 5.3 Conclusion

- Best 5-fold representation: `TF-IDF word + char`, F1 **0.9845**, ROC-AUC **0.9995**, PR-AUC **0.9986**.
- Word TF-IDF `(1, 2)` reached CV F1 **0.9738**.
- On validation, the combined representation reached F1 **0.9683** versus **0.9703**, but improved PR-AUC from **0.9983** to **0.9991** and reduced the ROC-AUC gap.

The predefined rule selects the cross-validation winner. Word n-grams capture meaning, while character n-grams help with misspellings, obfuscation, and unusual punctuation.

# 6. Feature Selection

Test methods appropriate for a non-negative sparse TF-IDF matrix:

- no additional selection as the control;
- `chi²` ranking by class association;
- L1-based selection using non-zero linear-model coefficients.

Correlation filters, ANOVA, and wrapper searches are not used because they scale poorly or add little value for this sparse high-dimensional representation.

In [ ]:
# Perform the analysis step below.
from sklearn.base import clone
from sklearn.feature_selection import SelectPercentile, chi2, SelectFromModel
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from sklearn.svm import LinearSVC

feature_selection_scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

feature_selection_candidates = {
    "No additional selection": "passthrough",
    "Chi2 top 25%": SelectPercentile(score_func=chi2, percentile=25),
    "Chi2 top 50%": SelectPercentile(score_func=chi2, percentile=50),
    "Chi2 top 75%": SelectPercentile(score_func=chi2, percentile=75),
    "L1 LinearSVC": SelectFromModel(
        LinearSVC(
            C=0.1,
            penalty="l1",
            dual=False,
            class_weight="balanced",
            max_iter=5000,
            random_state=0
        )
    )
}


In [ ]:
# Perform the analysis step below.
feature_selection_rows = []

for name, selector in feature_selection_candidates.items():
    pipeline = Pipeline([
        ("features", clone(text_representations[best_text_representation])),
        ("selector", clone(selector) if selector != "passthrough" else "passthrough"),
        ("model", make_logistic_regression())
    ])
    scores = cross_validate(
        pipeline,
        X_text_train,
        y_train_out,
        cv=cv,
        scoring=feature_selection_scoring,
        return_train_score=True,
        n_jobs=2,
        error_score="raise"
    )
    feature_selection_rows.append({
        "method": name,
        "cv_accuracy": scores["test_accuracy"].mean(),
        "cv_precision": scores["test_precision"].mean(),
        "cv_recall": scores["test_recall"].mean(),
        "cv_f1": scores["test_f1"].mean(),
        "cv_f1_std": scores["test_f1"].std(),
        "cv_roc_auc": scores["test_roc_auc"].mean(),
        "cv_pr_auc": scores["test_pr_auc"].mean(),
        "train_f1": scores["train_f1"].mean(),
        "f1_gap": scores["train_f1"].mean() - scores["test_f1"].mean()
    })

feature_selection_results = (
    pd.DataFrame(feature_selection_rows)
    .sort_values(["cv_f1", "cv_pr_auc"], ascending=False)
    .reset_index(drop=True)
)
best_feature_selector_name = feature_selection_results.loc[0, "method"]
best_feature_selector = feature_selection_candidates[best_feature_selector_name]

print("Best feature-selection strategy:", best_feature_selector_name)
feature_selection_results.round(4)


In [ ]:
# Perform the analysis step below.
feature_selection_plot = feature_selection_results.melt(
    id_vars="method",
    value_vars=["cv_f1", "cv_pr_auc"],
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(11, 6))
sns.barplot(
    data=feature_selection_plot,
    y="method",
    x="score",
    hue="metric"
)
plt.title("Feature selection with 5-fold cross-validation")
plt.xlim(0.90, 1.00)
plt.tight_layout()
plt.show()


## Feature-selection conclusion

- **No additional selection** won: CV F1 **0.9845**, PR-AUC **0.9986**.
- `Chi2 top 50%` was close but slightly worse at F1 **0.9835**.
- `Chi2 top 25%` fell to **0.9734** and L1 selection to **0.8819**.

TF-IDF is already constrained with `min_df`, `max_df`, and `max_features`. More aggressive selection removes useful n-grams, so the next stages use `passthrough`.

# 7. Classifier Comparison

Use the selected text representation and feature-selection strategy. Features and folds remain fixed while only the classifier changes.

## 7.1 Candidate models

Compare Dummy, Logistic Regression, Linear SVC, SGD, Ridge Classifier, and three Naive Bayes variants. The primary metric is F1; supporting metrics are accuracy, precision, recall, ROC-AUC, PR-AUC, fold variability, and train-CV gap.

In [ ]:
# Perform the analysis step below.
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.naive_bayes import BernoulliNB, ComplementNB, MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import make_scorer

model_scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}


In [ ]:
# Perform the analysis step below.
models_to_compare = {
    "Dummy Classifier": DummyClassifier(
        strategy="most_frequent",
        random_state=0
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=0
    ),
    "Linear SVC": LinearSVC(
        class_weight="balanced",
        random_state=0
    ),
    "SGD Classifier": SGDClassifier(
        loss="log_loss",
        max_iter=2000,
        class_weight="balanced",
        random_state=0
    ),
    "Multinomial NB": MultinomialNB(alpha=1.0),
    "Complement NB": ComplementNB(alpha=1.0),
    "Bernoulli NB": BernoulliNB(alpha=1.0),
    "Ridge Classifier": RidgeClassifier(
        class_weight="balanced"
    )
}

model_pipelines = {
    name: Pipeline([
        ("features", clone(text_representations[best_text_representation])),
        ("selector", clone(best_feature_selector) if best_feature_selector != "passthrough" else "passthrough"),
        ("model", model)
    ])
    for name, model in models_to_compare.items()
}

print("Number of models:", len(model_pipelines))


In [ ]:
# Perform the analysis step below.
model_comparison_rows = []

for name, pipeline in model_pipelines.items():
    scores = cross_validate(
        pipeline,
        X_text_train,
        y_train_out,
        cv=cv,
        scoring=model_scoring,
        return_train_score=True,
        n_jobs=2,
        error_score="raise"
    )

    model_comparison_rows.append({
        "model": name,
        "cv_accuracy": scores["test_accuracy"].mean(),
        "cv_precision": scores["test_precision"].mean(),
        "cv_recall": scores["test_recall"].mean(),
        "cv_f1": scores["test_f1"].mean(),
        "cv_f1_std": scores["test_f1"].std(),
        "train_f1": scores["train_f1"].mean(),
        "f1_gap": scores["train_f1"].mean() - scores["test_f1"].mean(),
        "cv_roc_auc": scores["test_roc_auc"].mean(),
        "cv_roc_auc_std": scores["test_roc_auc"].std(),
        "cv_pr_auc": scores["test_pr_auc"].mean(),
        "cv_pr_auc_std": scores["test_pr_auc"].std(),
        "train_roc_auc": scores["train_roc_auc"].mean(),
        "roc_auc_gap": (
            scores["train_roc_auc"].mean()
            - scores["test_roc_auc"].mean()
        ),
        "mean_fit_time": scores["fit_time"].mean()
    })

model_comparison_results = (
    pd.DataFrame(model_comparison_rows)
    .sort_values(["cv_f1", "cv_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

model_comparison_results.round(4)


In [ ]:
# Perform the analysis step below.
model_plot = model_comparison_results.melt(
    id_vars="model",
    value_vars=["cv_f1", "cv_roc_auc"],
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(12, 7))
sns.barplot(
    data=model_plot,
    y="model",
    x="score",
    hue="metric"
)
plt.title("Classifier comparison with 5-fold cross-validation")
plt.xlabel("Mean score")
plt.ylabel("Model")
plt.xlim(0, 1.0)
plt.tight_layout()
plt.show()

## 7.2 Validation check for the strongest models

In [ ]:
# Perform the analysis step below.
def get_model_score(pipeline, X):
    if hasattr(pipeline, "predict_proba"):
        return pipeline.predict_proba(X)[:, 1]
    return pipeline.decision_function(X)


top_model_names = model_comparison_results.head(3)["model"].tolist()
validation_model_rows = []

for name in top_model_names:
    pipeline = model_pipelines[name]
    pipeline.fit(X_text_train, y_train_out)

    train_prediction = pipeline.predict(X_text_train)
    validation_prediction = pipeline.predict(X_text_validation)
    train_score = get_model_score(pipeline, X_text_train)
    validation_score = get_model_score(pipeline, X_text_validation)

    validation_model_rows.append({
        "model": name,
        "train_accuracy": accuracy_score(y_train_out, train_prediction),
        "validation_accuracy": accuracy_score(y_validation_out, validation_prediction),
        "validation_precision": precision_score(
            y_validation_out,
            validation_prediction,
            zero_division=0
        ),
        "validation_recall": recall_score(
            y_validation_out,
            validation_prediction,
            zero_division=0
        ),
        "validation_f1": f1_score(
            y_validation_out,
            validation_prediction,
            zero_division=0
        ),
        "train_roc_auc": roc_auc_score(y_train_out, train_score),
        "validation_roc_auc": roc_auc_score(y_validation_out, validation_score),
        "validation_pr_auc": average_precision_score(
            y_validation_out, validation_score
        ),
        "roc_auc_gap": (
            roc_auc_score(y_train_out, train_score)
            - roc_auc_score(y_validation_out, validation_score)
        )
    })

model_validation_results = (
    pd.DataFrame(validation_model_rows)
    .sort_values("validation_f1", ascending=False)
    .reset_index(drop=True)
)

best_model_name = model_comparison_results.loc[0, "model"]
best_model_pipeline = model_pipelines[best_model_name]

print("Best model by CV F1:", best_model_name)
model_validation_results.round(4)


## 7.3 Conclusion

- Best algorithm: `Linear SVC`, CV F1 **0.9891**, accuracy **0.9946**, ROC-AUC **0.9998**, PR-AUC **0.9995**.
- On validation: accuracy **0.9968**, precision **0.9914**, recall **0.9957**, F1 **0.9935**.
- SGD followed at CV F1 **0.9887**, then Ridge at **0.9875**.
- Linear SVC train-CV F1 gap was **0.0109**: mild overfitting exists, but CV and validation remain strong and stable.
- Dummy F1 was **0**, confirming that trained models learn useful signal.

Linear SVC proceeds to hyperparameter tuning.

# 8. Hyperparameter Tuning

## 8.1 Search spaces

- `Linear SVC`: GridSearchCV over five `C` values and two class-weight settings;
- `Ridge Classifier`: GridSearchCV over four `alpha` values and two class-weight settings;
- `SGD Classifier`: RandomizedSearchCV over 12 sampled regularization, loss, averaging, and class-weight combinations.

All parameters are selected by mean CV F1 on training data only. Validation and the locked final test are excluded from search.

In [ ]:
# Perform the analysis step below.
import shutil
import tempfile

from joblib import Memory
from scipy.stats import loguniform
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


In [ ]:
# Perform the analysis step below.
tuning_cache_dir = tempfile.mkdtemp(prefix="email_tuning_")
tuning_memory = Memory(location=tuning_cache_dir, verbose=0)


def make_tuning_pipeline(model):
    return Pipeline(
        [
            ("features", clone(text_representations[best_text_representation])),
            ("selector", clone(best_feature_selector) if best_feature_selector != "passthrough" else "passthrough"),
            ("model", model)
        ],
        memory=tuning_memory
    )


linear_svc_search = GridSearchCV(
    estimator=make_tuning_pipeline(LinearSVC(random_state=0)),
    param_grid={
        "model__C": [0.03, 0.1, 0.3, 1, 3],
        "model__class_weight": [None, "balanced"]
    },
    scoring=model_scoring,
    refit="f1",
    cv=cv,
    return_train_score=True,
    n_jobs=2
)

ridge_search = GridSearchCV(
    estimator=make_tuning_pipeline(RidgeClassifier()),
    param_grid={
        "model__alpha": [0.1, 0.3, 1, 3],
        "model__class_weight": [None, "balanced"]
    },
    scoring=model_scoring,
    refit="f1",
    cv=cv,
    return_train_score=True,
    n_jobs=2
)

sgd_search = RandomizedSearchCV(
    estimator=make_tuning_pipeline(SGDClassifier(
        max_iter=2000,
        random_state=0
    )),
    param_distributions={
        "model__alpha": loguniform(1e-6, 1e-2),
        "model__loss": ["hinge", "log_loss", "modified_huber"],
        "model__penalty": ["l2", "l1", "elasticnet"],
        "model__class_weight": [None, "balanced"],
        "model__average": [False, True]
    },
    n_iter=12,
    scoring=model_scoring,
    refit="f1",
    cv=cv,
    return_train_score=True,
    random_state=0,
    n_jobs=2
)

coarse_searches = {
    "Linear SVC": linear_svc_search,
    "Ridge Classifier": ridge_search,
    "SGD Classifier": sgd_search
}

print("Linear SVC combinations:", 5 * 2)
print("Ridge combinations:", 4 * 2)
print("SGD sampled combinations:", 12)


In [ ]:
# Perform the analysis step below.
for name, search in coarse_searches.items():
    print(f"Parameter search: {name}")
    search.fit(X_text_train, y_train_out)
    print("Best CV F1:", round(search.best_score_, 4))
    print("Parameters:", search.best_params_)
    print()

tuning_searches = coarse_searches

# Perform the analysis step below.
for search in tuning_searches.values():
    search.best_estimator_.memory = None

tuning_memory.clear(warn=False)
shutil.rmtree(tuning_cache_dir, ignore_errors=True)


In [ ]:
# Perform the analysis step below.
tuning_rows = []

for name, search in tuning_searches.items():
    best_index = search.best_index_
    results = search.cv_results_
    coarse_search = coarse_searches[name]
    default_row = model_comparison_results.loc[
        model_comparison_results["model"] == name
    ].iloc[0]

    tuned_cv_f1 = results["mean_test_f1"][best_index]
    tuned_train_f1 = results["mean_train_f1"][best_index]
    tuned_cv_roc_auc = results["mean_test_roc_auc"][best_index]
    tuned_train_roc_auc = results["mean_train_roc_auc"][best_index]
    tuned_cv_pr_auc = results["mean_test_pr_auc"][best_index]

    tuning_rows.append({
        "model": name,
        "default_cv_f1": default_row["cv_f1"],
        "coarse_cv_f1": coarse_search.best_score_,
        "tuned_cv_f1": tuned_cv_f1,
        "cv_f1_change": tuned_cv_f1 - default_row["cv_f1"],
        "tuned_cv_f1_std": results["std_test_f1"][best_index],
        "tuned_cv_precision": results["mean_test_precision"][best_index],
        "tuned_cv_recall": results["mean_test_recall"][best_index],
        "default_cv_roc_auc": default_row["cv_roc_auc"],
        "tuned_cv_roc_auc": tuned_cv_roc_auc,
        "tuned_cv_pr_auc": tuned_cv_pr_auc,
        "cv_roc_auc_change": tuned_cv_roc_auc - default_row["cv_roc_auc"],
        "tuned_train_f1": tuned_train_f1,
        "tuned_f1_gap": tuned_train_f1 - tuned_cv_f1,
        "tuned_roc_auc_gap": tuned_train_roc_auc - tuned_cv_roc_auc,
        "best_params": search.best_params_
    })

tuning_results = (
    pd.DataFrame(tuning_rows)
    .sort_values("tuned_cv_f1", ascending=False)
    .reset_index(drop=True)
)

tuning_results.round(4)


In [ ]:
# Perform the analysis step below.
tuning_plot = tuning_results[[
    "model",
    "default_cv_f1",
    "coarse_cv_f1",
    "tuned_cv_f1"
]].melt(
    id_vars="model",
    var_name="version",
    value_name="cv_f1"
)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=tuning_plot,
    y="model",
    x="cv_f1",
    hue="version"
)
plt.title("CV F1 before and after tuning")
plt.xlabel("Mean CV F1")
plt.ylabel("Model")
plt.xlim(0.96, 1.00)
plt.tight_layout()
plt.show()


## 8.2 Validation check for tuned models

Validation did not participate in either search. Tuned candidates are compared once after parameter selection.

In [ ]:
# Perform the analysis step below.
tuned_validation_rows = []

for name, search in tuning_searches.items():
    pipeline = search.best_estimator_

    train_prediction = pipeline.predict(X_text_train)
    validation_prediction = pipeline.predict(X_text_validation)
    train_score = get_model_score(pipeline, X_text_train)
    validation_score = get_model_score(pipeline, X_text_validation)

    default_validation = model_validation_results.loc[
        model_validation_results["model"] == name
    ].iloc[0]

    tuned_train_f1 = f1_score(
        y_train_out,
        train_prediction,
        zero_division=0
    )
    tuned_validation_f1 = f1_score(
        y_validation_out,
        validation_prediction,
        zero_division=0
    )

    tuned_validation_rows.append({
        "model": name,
        "default_validation_f1": default_validation["validation_f1"],
        "tuned_validation_f1": tuned_validation_f1,
        "validation_f1_change": (
            tuned_validation_f1
            - default_validation["validation_f1"]
        ),
        "tuned_validation_accuracy": accuracy_score(
            y_validation_out,
            validation_prediction
        ),
        "tuned_validation_precision": precision_score(
            y_validation_out,
            validation_prediction,
            zero_division=0
        ),
        "tuned_validation_recall": recall_score(
            y_validation_out,
            validation_prediction,
            zero_division=0
        ),
        "tuned_train_f1": tuned_train_f1,
        "tuned_train_validation_f1_gap": (
            tuned_train_f1 - tuned_validation_f1
        ),
        "tuned_validation_roc_auc": roc_auc_score(
            y_validation_out, validation_score
        ),
        "tuned_validation_pr_auc": average_precision_score(
            y_validation_out, validation_score
        ),
        "tuned_roc_auc_gap": (
            roc_auc_score(y_train_out, train_score)
            - roc_auc_score(y_validation_out, validation_score)
        )
    })

tuned_validation_results = (
    pd.DataFrame(tuned_validation_rows)
    .sort_values("tuned_validation_f1", ascending=False)
    .reset_index(drop=True)
)

best_tuned_model_name = tuning_results.loc[0, "model"]
best_tuned_pipeline = tuning_searches[best_tuned_model_name].best_estimator_

print("Best tuned model by CV F1:", best_tuned_model_name)
tuned_validation_results.round(4)


## 8.3 Conclusion

- Linear SVC retained the baseline parameters `C=1`, `class_weight='balanced'`, CV F1 **0.9891**.
- Ridge also did not improve: `alpha=1`, `class_weight='balanced'`, CV F1 **0.9875**.
- Tuned SGD reached CV F1 **0.9848**, **0.0040** below default, although validation F1 rose to **0.9914**.
- No critical overfitting is visible; ROC-AUC gaps are **0.0001–0.0003**.

Tuning did not beat the default Linear SVC, so the simpler winning single model is retained.

# 9. Ensemble Evaluation

Evaluate two practical soft-voting ensembles that combine strong linear and probabilistic models. Preprocessing and feature selection remain inside each pipeline.

In [ ]:
# Perform the analysis step below.
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier


def make_soft_voting(weights=None):
    return VotingClassifier(
        estimators=[
            (
                "svc",
                CalibratedClassifierCV(
                    LinearSVC(C=5.0, random_state=0),
                    method="sigmoid",
                    cv=3
                )
            ),
            (
                "sgd",
                SGDClassifier(
                    loss="log_loss",
                    alpha=0.00019089171584870895,
                    penalty="elasticnet",
                    class_weight="balanced",
                    max_iter=3000,
                    random_state=0
                )
            ),
            (
                "lr",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=0
                )
            )
        ],
        voting="soft",
        weights=weights,
        n_jobs=2
    )


ensemble_pipelines = {
    "Soft Voting equal": Pipeline([
        ("features", clone(text_representations[best_text_representation])),
        ("selector", clone(best_feature_selector) if best_feature_selector != "passthrough" else "passthrough"),
        ("model", make_soft_voting())
    ]),
    "Soft Voting weighted": Pipeline([
        ("features", clone(text_representations[best_text_representation])),
        ("selector", clone(best_feature_selector) if best_feature_selector != "passthrough" else "passthrough"),
        ("model", make_soft_voting(weights=[2, 2, 1]))
    ])
}


In [ ]:
# Perform the analysis step below.
ensemble_rows = []
for name, pipeline in ensemble_pipelines.items():
    scores = cross_validate(
        pipeline,
        X_text_train,
        y_train_out,
        cv=cv,
        scoring=model_scoring,
        return_train_score=True,
        n_jobs=2,
        error_score="raise"
    )
    ensemble_rows.append({
        "model": name,
        "cv_accuracy": scores["test_accuracy"].mean(),
        "cv_precision": scores["test_precision"].mean(),
        "cv_recall": scores["test_recall"].mean(),
        "cv_f1": scores["test_f1"].mean(),
        "cv_f1_std": scores["test_f1"].std(),
        "cv_roc_auc": scores["test_roc_auc"].mean(),
        "cv_pr_auc": scores["test_pr_auc"].mean(),
        "train_f1": scores["train_f1"].mean(),
        "f1_gap": scores["train_f1"].mean() - scores["test_f1"].mean()
    })

ensemble_results = (
    pd.DataFrame(ensemble_rows)
    .sort_values(["cv_f1", "cv_pr_auc"], ascending=False)
    .reset_index(drop=True)
)
ensemble_results.round(4)


In [ ]:
# Perform the analysis step below.
best_ensemble_name = ensemble_results.loc[0, "model"]
best_ensemble_pipeline = ensemble_pipelines[best_ensemble_name]
best_ensemble_pipeline.fit(X_text_train, y_train_out)

ensemble_validation_prediction = best_ensemble_pipeline.predict(
    X_text_validation
)
ensemble_validation_score = best_ensemble_pipeline.predict_proba(
    X_text_validation
)[:, 1]

ensemble_validation_result = pd.DataFrame([{
    "model": best_ensemble_name,
    "validation_accuracy": accuracy_score(
        y_validation_out, ensemble_validation_prediction
    ),
    "validation_precision": precision_score(
        y_validation_out, ensemble_validation_prediction
    ),
    "validation_recall": recall_score(
        y_validation_out, ensemble_validation_prediction
    ),
    "validation_f1": f1_score(
        y_validation_out, ensemble_validation_prediction
    ),
    "validation_roc_auc": roc_auc_score(
        y_validation_out, ensemble_validation_score
    ),
    "validation_pr_auc": average_precision_score(
        y_validation_out, ensemble_validation_score
    )
}])

ensemble_validation_result.round(4)


## Ensemble conclusion

- Best ensemble: `Soft Voting equal`, CV F1 **0.9887**, PR-AUC **0.9992**.
- Validation F1 was **0.9850**.
- Linear SVC remained better: CV F1 **0.9891**, validation F1 **0.9935**.

The ensemble was tested empirically but rejected because added complexity did not improve quality.

# 10. PR-AUC and Decision-Threshold Selection

PR-AUC complements ROC-AUC because spam is the minority class. The threshold is selected on validation only. High precision is prioritized so legitimate mail is rarely sent to spam.

In [ ]:
# Perform the analysis step below.
import sys
from pathlib import Path
from sklearn.metrics import precision_recall_curve
from sklearn.preprocessing import FunctionTransformer

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "Email_Spam"
    else Path.cwd()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Email_Spam.email_spam_pipeline import clean_email_series

best_single_name = tuning_results.loc[0, "model"]
best_single_cv_f1 = tuning_results.loc[0, "tuned_cv_f1"]
best_ensemble_cv_f1 = ensemble_results.loc[0, "cv_f1"]

if best_ensemble_cv_f1 > best_single_cv_f1:
    selected_model_name = best_ensemble_name
    selected_clean_pipeline = clone(best_ensemble_pipeline)
    selected_cv_f1 = best_ensemble_cv_f1
else:
    selected_model_name = best_single_name
    selected_clean_pipeline = clone(best_tuned_pipeline)
    selected_cv_f1 = best_single_cv_f1

locked_raw_pipeline = Pipeline([
    ("cleaner", FunctionTransformer(clean_email_series, validate=False)),
    ("features", clone(selected_clean_pipeline.named_steps["features"])),
    ("selector", clone(selected_clean_pipeline.named_steps["selector"]) if selected_clean_pipeline.named_steps["selector"] != "passthrough" else "passthrough"),
    ("model", clone(selected_clean_pipeline.named_steps["model"]))
])

X_train_raw = clean_data.loc[train_index, "original_text"]
X_validation_raw = clean_data.loc[validation_index, "original_text"]
locked_raw_pipeline.fit(X_train_raw, y_train_out)

validation_scores_for_threshold = get_model_score(
    locked_raw_pipeline, X_validation_raw
)

print("Selected model:", selected_model_name)
print("Train CV F1:", round(float(selected_cv_f1), 4))


In [ ]:
# Perform the analysis step below.
pr_precision, pr_recall, pr_thresholds = precision_recall_curve(
    y_validation_out,
    validation_scores_for_threshold
)

threshold_results = pd.DataFrame({
    "threshold": pr_thresholds,
    "precision": pr_precision[:-1],
    "recall": pr_recall[:-1]
})
threshold_results["f1"] = (
    2 * threshold_results["precision"] * threshold_results["recall"]
    / (
        threshold_results["precision"]
        + threshold_results["recall"]
    ).replace(0, np.nan)
)

precision_target = 0.99
eligible_thresholds = threshold_results.loc[
    threshold_results["precision"] >= precision_target
]

if len(eligible_thresholds):
    selected_threshold_row = eligible_thresholds.sort_values(
        ["recall", "f1", "threshold"],
        ascending=[False, False, True]
    ).iloc[0]
else:
    selected_threshold_row = threshold_results.sort_values(
        ["f1", "precision"],
        ascending=False
    ).iloc[0]

selected_threshold = float(selected_threshold_row["threshold"])
validation_prediction_at_threshold = (
    validation_scores_for_threshold >= selected_threshold
).astype(int)

threshold_summary = pd.DataFrame([{
    "selected_threshold": selected_threshold,
    "validation_accuracy": accuracy_score(
        y_validation_out, validation_prediction_at_threshold
    ),
    "validation_precision": precision_score(
        y_validation_out,
        validation_prediction_at_threshold,
        zero_division=0
    ),
    "validation_recall": recall_score(
        y_validation_out,
        validation_prediction_at_threshold,
        zero_division=0
    ),
    "validation_f1": f1_score(
        y_validation_out,
        validation_prediction_at_threshold,
        zero_division=0
    ),
    "validation_pr_auc": average_precision_score(
        y_validation_out, validation_scores_for_threshold
    )
}])

threshold_summary.round(4)


In [ ]:
# Perform the analysis step below.
plt.figure(figsize=(8, 6))
plt.plot(pr_recall, pr_precision, label="Validation PR curve")
plt.scatter(
    selected_threshold_row["recall"],
    selected_threshold_row["precision"],
    color="red",
    s=70,
    label=f"Threshold = {selected_threshold:.4f}"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curve on validation")
plt.xlim(0, 1.02)
plt.ylim(0, 1.02)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## Threshold conclusion

The selected validation threshold is **0.0676**, maximizing recall subject to precision of at least 0.99. Validation precision, recall, and F1 are each **0.9957**, with PR-AUC **0.9998**. The threshold is then frozen and not adjusted on the locked final test.

# 11. Independent Final Test and Error Analysis

The locked final test was separated at the beginning using normalized-text groups. Identical messages therefore cannot appear in both development and final test data. Neither final-test rows nor labels were used in EDA, feature engineering, feature selection, cross-validation, tuning, ensemble selection, or threshold selection.

In [ ]:
# Perform the analysis step below.
final_test_data = locked_final_raw.copy()
final_test_data["original_text"] = (
    final_test_data["text"].fillna("").astype(str)
)
final_test_data = final_test_data.loc[
    final_test_data["original_text"].str.strip().ne("")
].copy()
final_test_data = final_test_data.drop_duplicates(
    subset="original_text"
).reset_index(drop=True)
final_test_data["clean_text"] = final_test_data["original_text"].map(
    clean_email_text
)
final_test_data = final_test_data.drop_duplicates(
    subset="clean_text"
).reset_index(drop=True)

overlap = set(clean_data["clean_text"]) & set(final_test_data["clean_text"])
if overlap:
    raise ValueError("Development and final test contain overlapping normalized texts")

X_final_test = final_test_data["original_text"]
y_final_test = final_test_data["spam"]

final_split_summary = pd.DataFrame({
    "sample": ["Train", "Validation", "Locked final test"],
    "rows": [len(X_train_raw), len(X_validation_raw), len(X_final_test)],
    "spam_share": [
        y_train_out.mean(),
        y_validation_out.mean(),
        y_final_test.mean()
    ]
})

print("Normalized-text overlap:", len(overlap))
final_split_summary.round(4)


In [ ]:
# Perform the analysis step below.
from sklearn.metrics import classification_report, confusion_matrix


def evaluate_locked_sample(sample_name, X, y):
    score = get_model_score(locked_raw_pipeline, X)
    prediction = (score >= selected_threshold).astype(int)
    return {
        "sample": sample_name,
        "accuracy": accuracy_score(y, prediction),
        "precision": precision_score(y, prediction, zero_division=0),
        "recall": recall_score(y, prediction, zero_division=0),
        "f1": f1_score(y, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y, score),
        "pr_auc": average_precision_score(y, score)
    }, prediction, score


train_result, train_final_prediction, train_final_score = evaluate_locked_sample(
    "train", X_train_raw, y_train_out
)
validation_result, validation_final_prediction, validation_final_score = (
    evaluate_locked_sample(
        "validation", X_validation_raw, y_validation_out
    )
)
test_result, final_test_prediction, final_test_score = evaluate_locked_sample(
    "locked_final_test", X_final_test, y_final_test
)

final_metrics = pd.DataFrame([
    train_result,
    validation_result,
    test_result
])
final_metrics.round(4)


In [ ]:
# Perform the analysis step below.
final_confusion = confusion_matrix(
    y_final_test,
    final_test_prediction
)

plt.figure(figsize=(6, 5))
sns.heatmap(
    final_confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Legitimate email", "Spam"],
    yticklabels=["Legitimate email", "Spam"]
)
plt.title("Confusion matrix — locked final test")
plt.xlabel("Prediction")
plt.ylabel("Actual class")
plt.tight_layout()
plt.show()


## Error analysis

Inspect false positives, false negatives, message length, and frequent error vocabulary. This analysis explains failures but is not used to retune the model or threshold.

In [ ]:
# Perform the analysis step below.
final_error_analysis = pd.DataFrame({
    "text": X_final_test.to_numpy(),
    "actual": y_final_test.to_numpy(),
    "predicted": final_test_prediction,
    "score": final_test_score
})
final_error_analysis["error_type"] = np.select(
    [
        (final_error_analysis["actual"] == 0)
        & (final_error_analysis["predicted"] == 1),
        (final_error_analysis["actual"] == 1)
        & (final_error_analysis["predicted"] == 0)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)
final_error_analysis["text_length"] = final_error_analysis["text"].str.len()
final_error_analysis["absolute_margin"] = (
    final_error_analysis["score"] - selected_threshold
).abs()

error_examples = (
    final_error_analysis.loc[
        final_error_analysis["error_type"] != "Correct"
    ]
    .sort_values("absolute_margin", ascending=False)
    .assign(
        text_preview=lambda frame: frame["text"]
        .str.replace(r"\s+", " ", regex=True)
        .str.slice(0, 240)
    )
    [[
        "error_type", "actual", "predicted", "score",
        "text_length", "text_preview"
    ]]
)
error_examples.round(4)


In [ ]:
# Perform the analysis step below.
final_error_analysis["length_group"] = pd.qcut(
    final_error_analysis["text_length"],
    q=4,
    duplicates="drop"
)
length_error_analysis = (
    final_error_analysis.groupby("length_group", observed=True)
    .agg(
        emails=("actual", "size"),
        errors=("error_type", lambda values: (values != "Correct").sum()),
        spam_share=("actual", "mean"),
        average_score=("score", "mean")
    )
    .reset_index()
)
length_error_analysis["error_rate"] = (
    length_error_analysis["errors"]
    / length_error_analysis["emails"]
)
length_error_analysis.round(4)


In [ ]:
# Perform the analysis step below.
error_term_rows = []
for error_type in ["False Positive", "False Negative"]:
    error_texts = final_error_analysis.loc[
        final_error_analysis["error_type"] == error_type,
        "text"
    ]
    if len(error_texts) == 0:
        continue
    error_vectorizer = CountVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
        max_features=1000
    )
    error_matrix = error_vectorizer.fit_transform(error_texts)
    error_counts = np.asarray(error_matrix.sum(axis=0)).ravel()
    error_terms = error_vectorizer.get_feature_names_out()
    for term_index in error_counts.argsort()[::-1][:12]:
        error_term_rows.append({
            "error_type": error_type,
            "term": error_terms[term_index],
            "count": error_counts[term_index]
        })

error_terms_table = pd.DataFrame(error_term_rows)
error_terms_table


## Detailed review of every misclassified email

Review all seven errors individually, including actual and predicted class, score relative to the threshold, length, links, and recognizable signals.

In [ ]:
# Perform the analysis step below.
import re


def find_spam_signals(text):
    text_lower = str(text).lower()
    signal_patterns = {
        "free": r"\bfree\b",
        "money/offer": r"\b(money|offer|discount|prize|win|won|cash)\b",
        "call to action": r"\b(click|buy|order|subscribe|register|download)\b",
        "link": r"https?\s*:\s*/\s*/|www\s*\.",
        "strong punctuation": r"!{2,}|\?{2,}",
    }
    found = [
        name for name, pattern in signal_patterns.items()
        if re.search(pattern, text_lower)
    ]
    return ", ".join(found) if found else "few explicit spam signals"


def explain_possible_error(row):
    text_lower = str(row["text"]).lower()
    if row["error_type"] == "False Positive":
        if re.search(r"\b(free|download|business|archive|information)\b", text_lower):
            return (
                "The legitimate email contains promotional or internet vocabulary "
                "that frequently appeared in spam."
            )
        return "Its vocabulary made the legitimate email resemble spam."
    if row["text_length"] < 150:
        return (
            "Very short spam with too little context or distinctive vocabulary "
            "for a confident decision."
        )
    if row["text_length"] > 3000:
        return (
            "A long technical or academic email resembles "
            "ordinary professional correspondence."
        )
    return "The spam is neutrally written and resembles a normal business message."


detailed_error_cases = (
    final_error_analysis.loc[
        final_error_analysis["error_type"] != "Correct"
    ]
    .copy()
    .assign(
        case_id=lambda frame: frame.index + 1,
        actual_label=lambda frame: frame["actual"].map({0: "Legitimate", 1: "Spam"}),
        predicted_label=lambda frame: frame["predicted"].map({0: "Legitimate", 1: "Spam"}),
        threshold=selected_threshold,
        threshold_margin=lambda frame: frame["score"] - selected_threshold,
        url_count=lambda frame: frame["text"].str.count(
            r"(?i)https?\s*:\s*/\s*/|www\s*\."
        ),
        subject=lambda frame: (
            frame["text"]
            .str.replace(r"\s+", " ", regex=True)
            .str.extract(r"(?i)subject\s*:\s*(.{1,110})", expand=False)
            .fillna("Subject not found")
        ),
        spam_signals=lambda frame: frame["text"].map(find_spam_signals),
    )
)
detailed_error_cases["possible_reason"] = detailed_error_cases.apply(
    explain_possible_error,
    axis=1,
)

error_case_table = detailed_error_cases[[
    "case_id", "error_type", "actual_label", "predicted_label",
    "score", "threshold", "threshold_margin", "text_length",
    "url_count", "spam_signals", "possible_reason", "subject"
]].sort_values(["error_type", "score"], ascending=[True, False])

error_case_table.round(4)


In [ ]:
# Perform the analysis step below.
from IPython.display import display

for _, error_row in detailed_error_cases.sort_values(
    ["error_type", "score"],
    ascending=[True, False],
).iterrows():
    print("=" * 100)
    print(
        f"Case {int(error_row['case_id'])} | {error_row['error_type']} | "
        f"Actual: {error_row['actual_label']} | "
        f"Predicted: {error_row['predicted_label']}"
    )
    print(
        f"Score: {error_row['score']:.4f} | "
        f"Threshold: {selected_threshold:.4f} | "
        f"Difference: {error_row['threshold_margin']:+.4f} | "
        f"Length: {int(error_row['text_length'])} characters"
    )
    print(f"Signals: {error_row['spam_signals']}")
    print(f"Possible reason: {error_row['possible_reason']}")
    print("Email excerpt:")
    print(re.sub(r"\s+", " ", str(error_row["text"]))[:700])
    print()

error_type_summary = (
    detailed_error_cases.groupby("error_type")
    .agg(
        emails=("actual", "size"),
        average_score=("score", "mean"),
        median_length=("text_length", "median"),
        average_links=("url_count", "mean"),
    )
    .round(4)
)
display(error_type_summary)


### What the seven errors reveal

1. Two very short false negatives contained only 81 and 76 characters, providing little context or spam vocabulary.
2. Three long false negatives resembled professional material: a call for papers, software information, and a road-safety message. Academic and technical style masked the spam signal.
3. The road-safety email scored **0.0557** against a threshold of **0.0676**, making it a borderline case.
4. One false positive was a legitimate FT news-archive message containing domains, URLs, and business vocabulary. Its score of **0.4486** shows a confident mistake.
5. Another false positive contained `free latex`, a URL, and `download`; its score of **0.0925** was only slightly above the threshold and the source label may be ambiguous.
6. The longest-message group contained **3 errors among 206 messages (1.46%)**, but seven total errors are insufficient to claim that length alone is the main cause.

The difficult scenarios are short context-poor spam and long professionally written spam. Legitimate messages with URLs and promotional vocabulary can create false positives.

## Final conclusion

- Locked final test: **825** emails, spam rate **24.85%**, normalized-text overlap with development **0**.
- Accuracy **0.9915**, precision **0.9901**, recall **0.9756**, F1 **0.9828**.
- ROC-AUC **0.9996**, PR-AUC **0.9988**.
- Errors: **2 false positives** and **5 false negatives**.

The model separates spam effectively and rarely sends legitimate mail to spam. The reduction from validation F1 demonstrates the value of an untouched final test. The model and threshold are not changed after error analysis.

In [ ]:
# Perform the analysis step below.
import joblib

X_deployment = pd.concat([X_train_raw, X_validation_raw])
y_deployment = pd.concat([y_train_out, y_validation_out])
deployment_pipeline = clone(locked_raw_pipeline)
deployment_pipeline.fit(X_deployment, y_deployment)

final_model_artifact = {
    "pipeline": deployment_pipeline,
    "decision_threshold": selected_threshold,
    "positive_class": "spam",
    "selected_model": selected_model_name,
    "text_representation": best_text_representation,
    "feature_selection": best_feature_selector_name,
    "final_test_metrics": test_result
}

final_model_path = (
    project_root / "Email_Spam" / "Email_Spam_Final_Pipeline.pkl"
)
joblib.dump(final_model_artifact, final_model_path)

# Perform the analysis step below.
loaded_artifact = joblib.load(final_model_path)
example_emails = [
    "Free money! Click here now to claim your prize.",
    "Hi team, the meeting starts tomorrow at 10 AM."
]
example_scores = get_model_score(
    loaded_artifact["pipeline"], example_emails
)
example_predictions = (
    example_scores >= loaded_artifact["decision_threshold"]
).astype(int)

print("Model saved:", final_model_path)
print("Selected threshold:", round(selected_threshold, 6))
print("Example predictions:", example_predictions.tolist())
